In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# Inference for `v6_dinov2_lss_bev_cleaned` and `v7_rtmdet_cspnext_lss_bev_cleaned`

Notebook для GPU-инференса двух run'ов:
- `best.pt` и `ema_best.pt` для `v6`
- `best.pt` и `ema_best.pt` для `v7`
- подбор global threshold на `val`
- сохранение `val/test` probabilities для локального CPU-ансамбля
- опциональная сборка single-model `zip`-посылок


In [1]:
import os
import gc
import json
import math
import random
import hashlib
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile, ImageFilter
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

DATA_ROOT = Path('/home/jupyter/project')
DATA_TRAIN = DATA_ROOT / 'autonomy_yandex_dataset_train'
DATA_VAL = DATA_ROOT / 'autonomy_yandex_dataset_val'
DATA_TEST = DATA_ROOT / 'autonomy_yandex_dataset_test'

V6_RUN_DIR = Path('/home/jupyter/project/pray/kaggle_kernel_output/runs/v6_dinov2_lss_bev_cleaned')
V7_RUN_DIR = Path('/home/jupyter/project/runs/v7_rtmdet_cspnext_lss_bev_cleaned')
OUT_DIR = Path('/home/jupyter/project/ensemble_infer_outputs_v6_v7')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_RTMDET_CKPT = DATA_ROOT / 'rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth'

RUN_SPECS = [
    {'name': 'v6_dino', 'run_dir': V6_RUN_DIR},
    {'name': 'v7_rtmdet', 'run_dir': V7_RUN_DIR},
]

for p in [DATA_TRAIN, DATA_VAL, DATA_TEST, V6_RUN_DIR, V7_RUN_DIR]:
    assert p.exists(), p

print('OUT_DIR:', OUT_DIR)
print('DEFAULT_RTMDET_CKPT exists:', DEFAULT_RTMDET_CKPT.exists())


device: cuda
gpu: Tesla T4
OUT_DIR: /home/jupyter/project/ensemble_infer_outputs_v6_v7
DEFAULT_RTMDET_CKPT exists: True


In [2]:
CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def kaggle_safe_name(name: str) -> str:
    return name.replace(':', '_')


def resolve_info_path(base_dir: Path, p) -> Path:
    p = Path(str(p))
    candidates = []

    candidates.append(p)
    candidates.append(base_dir / p)
    candidates.append(base_dir.parent / p)

    parts = list(p.parts)
    for anchor in [
        'autonomy_yandex_dataset_train',
        'autonomy_yandex_dataset_val',
        'autonomy_yandex_dataset_test',
        'autonomy_yandex_dataset_train_kaggle_safe',
        'autonomy_yandex_dataset_val_kaggle_safe',
        'autonomy_yandex_dataset_test_kaggle_safe',
    ]:
        if anchor in parts:
            i = parts.index(anchor)
            rel = Path(*parts[i + 1:])
            candidates.append(base_dir / rel)
            candidates.append(base_dir.parent / rel)
            safe_rel = Path(*[kaggle_safe_name(x) for x in rel.parts])
            candidates.append(base_dir / safe_rel)
            candidates.append(base_dir.parent / safe_rel)

    safe_p = Path(*[kaggle_safe_name(x) for x in parts])
    candidates.append(base_dir / safe_p)
    candidates.append(base_dir.parent / safe_p)

    seen = set()
    for cand in candidates:
        cand = Path(cand)
        key = str(cand)
        if key in seen:
            continue
        seen.add(key)
        if cand.exists():
            return cand

    raise FileNotFoundError(f'Could not resolve path: {p} from base_dir={base_dir}')


def load_info_with_root(data_dir: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(data_dir / 'info.csv', index_col=0).reset_index(drop=True).copy()
    df['__data_root'] = str(data_dir)
    df['__source_split'] = split_name
    return df


def remap_kaggle_paths(df: pd.DataFrame, train_dir: Path, val_dir: Path, test_dir: Path) -> pd.DataFrame:
    df = df.copy()
    path_cols = [*CAMERA_NAMES, *INTRINSICS_NAMES, *CAR2CAM_NAMES, GT_NAME]

    def pick_root(split: str) -> Path:
        if split == 'train':
            return train_dir
        if split == 'val':
            return val_dir
        if split == 'test':
            return test_dir
        return train_dir

    def rewrite_path(p, split: str):
        if pd.isna(p):
            return p
        p = str(p)
        pp = Path(p)
        if pp.exists():
            return str(pp)

        root = pick_root(str(split))
        parts = list(Path(p).parts)

        for anchor in [
            'autonomy_yandex_dataset_train',
            'autonomy_yandex_dataset_val',
            'autonomy_yandex_dataset_test',
            'autonomy_yandex_dataset_train_kaggle_safe',
            'autonomy_yandex_dataset_val_kaggle_safe',
            'autonomy_yandex_dataset_test_kaggle_safe',
        ]:
            if anchor in parts:
                i = parts.index(anchor)
                rel = Path(*parts[i + 1:])
                cand = root / rel
                if cand.exists():
                    return str(cand)
                safe_rel = Path(*[kaggle_safe_name(x) for x in rel.parts])
                cand = root / safe_rel
                if cand.exists():
                    return str(cand)

        safe_parts = [kaggle_safe_name(x) for x in parts]
        safe_p = Path(*safe_parts)
        cand = root / safe_p
        if cand.exists():
            return str(cand)

        cand = root / kaggle_safe_name(Path(p).name)
        if cand.exists():
            return str(cand)

        return p

    if '__source_split' in df.columns:
        df['__data_root'] = df['__source_split'].map({
            'train': str(train_dir),
            'val': str(val_dir),
            'test': str(test_dir),
        }).fillna(str(train_dir))
    else:
        df['__data_root'] = str(train_dir)

    for col in path_cols:
        if col in df.columns:
            df[col] = [
                rewrite_path(p, split)
                for p, split in zip(df[col], df.get('__source_split', pd.Series(['train'] * len(df))))
            ]

    return df


def resolve_row_path(row: pd.Series, key: str) -> Path:
    return resolve_info_path(Path(row['__data_root']), row[key])


In [3]:
def pick_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError(paths)

MERGED_CLEANED_CSV = pick_existing([
    V6_RUN_DIR / 'preproc_cache' / 'merged_cleaned.csv',
    V7_RUN_DIR / 'preproc_cache' / 'merged_cleaned.csv',
    V6_RUN_DIR / 'merged_cleaned.csv',
    V7_RUN_DIR / 'merged_cleaned.csv',
])
SPLIT_NPZ = pick_existing([
    V6_RUN_DIR / 'preproc_cache' / 'test_matched_val200_split.npz',
    V7_RUN_DIR / 'preproc_cache' / 'test_matched_val200_split.npz',
    V6_RUN_DIR / 'test_matched_val200_split.npz',
    V7_RUN_DIR / 'test_matched_val200_split.npz',
])

clean_info = pd.read_csv(MERGED_CLEANED_CSV)
split = np.load(SPLIT_NPZ)
train_idx = split['train_idx'].tolist()
val_idx = split['val_idx'].tolist()

clean_info = remap_kaggle_paths(clean_info, DATA_TRAIN, DATA_VAL, DATA_TEST)
train_info = remap_kaggle_paths(clean_info.iloc[train_idx].reset_index(drop=True).copy(), DATA_TRAIN, DATA_VAL, DATA_TEST)
val_info = remap_kaggle_paths(clean_info.iloc[val_idx].reset_index(drop=True).copy(), DATA_TRAIN, DATA_VAL, DATA_TEST)
test_info = remap_kaggle_paths(load_info_with_root(DATA_TEST, 'test'), DATA_TRAIN, DATA_VAL, DATA_TEST)

print('merged_cleaned:', MERGED_CLEANED_CSV)
print('split_npz:', SPLIT_NPZ)
print('train:', len(train_info), 'val:', len(val_info), 'test:', len(test_info))

val_info.to_csv(OUT_DIR / 'val_info_export.csv', index=False)
test_info.to_csv(OUT_DIR / 'test_info_export.csv', index=False)
(DATA_TEST / 'info.csv').replace(OUT_DIR / 'test_info_official.csv') if False else None
(OUT_DIR / 'test_info_official.csv').write_bytes((DATA_TEST / 'info.csv').read_bytes())


merged_cleaned: /home/jupyter/project/pray/kaggle_kernel_output/runs/v6_dinov2_lss_bev_cleaned/preproc_cache/merged_cleaned.csv
split_npz: /home/jupyter/project/pray/kaggle_kernel_output/runs/v6_dinov2_lss_bev_cleaned/preproc_cache/test_matched_val200_split.npz
train: 4273 val: 220 test: 2000


2904011

In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.data import BEVDatasetV4Clean


In [6]:
def gn_groups(channels: int, requested: int = 8) -> int:
    g = min(requested, channels)
    while channels % g != 0 and g > 1:
        g -= 1
    return max(g, 1)


class ConvGNAct(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1, groups=8, act=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_c, out_c, k, stride=s, padding=p, bias=False),
            nn.GroupNorm(gn_groups(out_c, groups), out_c),
        ]
        if act:
            layers.append(nn.SiLU(inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class ResidualBlock2d(nn.Module):
    def __init__(self, in_c, out_c, stride=1, groups=8):
        super().__init__()
        self.conv1 = ConvGNAct(in_c, out_c, k=3, s=stride, p=1, groups=groups, act=True)
        self.conv2 = ConvGNAct(out_c, out_c, k=3, s=1, p=1, groups=groups, act=False)
        if stride != 1 or in_c != out_c:
            self.skip = ConvGNAct(in_c, out_c, k=1, s=stride, p=0, groups=groups, act=False)
        else:
            self.skip = nn.Identity()
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.conv2(self.conv1(x)) + self.skip(x))


class ASPP2d(nn.Module):
    def __init__(self, in_c, out_c, rates=(1, 3, 6), groups=8):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=r, dilation=r, bias=False),
                nn.GroupNorm(gn_groups(out_c, groups), out_c),
                nn.SiLU(inplace=True),
            )
            for r in rates
        ])
        self.proj = ConvGNAct(out_c * len(rates), out_c, k=1, s=1, p=0, groups=groups, act=True)

    def forward(self, x):
        xs = [b(x) for b in self.branches]
        return self.proj(torch.cat(xs, dim=1))


class ConvBNAct(nn.Module):
    def __init__(self, in_c, out_c, k, stride=1, padding=None, groups=1,
                 bias=False, eps=1e-3, momentum=0.01, act=True):
        super().__init__()
        if padding is None:
            padding = k // 2
        self.conv = nn.Conv2d(in_c, out_c, k, stride=stride, padding=padding, groups=groups, bias=bias)
        self.bn = nn.BatchNorm2d(out_c, eps=eps, momentum=momentum)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class ChannelAttention(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Conv2d(channels, channels, 1, 1, 0, bias=True)
        self.act = nn.Hardsigmoid(inplace=True)

    def forward(self, x):
        with torch.cuda.amp.autocast(enabled=False):
            a = self.pool(x.float())
            a = self.fc(a)
            a = self.act(a)
        return x * a.to(dtype=x.dtype)


class SPPBottleneck(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes=(5, 9, 13), eps=1e-3, momentum=0.01):
        super().__init__()
        mid_channels = in_channels // 2
        self.conv1 = ConvBNAct(in_channels, mid_channels, 1, eps=eps, momentum=momentum)
        self.poolings = nn.ModuleList([
            nn.MaxPool2d(kernel_size=ks, stride=1, padding=ks // 2) for ks in kernel_sizes
        ])
        self.conv2 = ConvBNAct(mid_channels * (len(kernel_sizes) + 1), out_channels, 1, eps=eps, momentum=momentum)

    def forward(self, x):
        x = self.conv1(x)
        with torch.cuda.amp.autocast(enabled=False):
            x32 = x.float()
            x = torch.cat([x32] + [pool(x32) for pool in self.poolings], dim=1)
        x = x.to(dtype=self.conv2.conv.weight.dtype)
        return self.conv2(x)


class CSPNeXtBlock(nn.Module):
    def __init__(self, in_channels, out_channels, expansion=0.5, add_identity=True, kernel_size=5,
                 eps=1e-3, momentum=0.01):
        super().__init__()
        hidden_channels = int(out_channels * expansion)
        self.conv1 = ConvBNAct(in_channels, hidden_channels, 3, eps=eps, momentum=momentum)
        self.conv2 = nn.Sequential(
            ConvBNAct(hidden_channels, hidden_channels, kernel_size, groups=hidden_channels, eps=eps, momentum=momentum),
            ConvBNAct(hidden_channels, out_channels, 1, eps=eps, momentum=momentum),
        )
        self.add_identity = add_identity and in_channels == out_channels

    def forward(self, x):
        y = self.conv1(x)
        y = self.conv2(y)
        return y + x if self.add_identity else y


class CSPLayer(nn.Module):
    def __init__(self, in_channels, out_channels, expand_ratio=0.5, num_blocks=1,
                 add_identity=True, channel_attention=True, eps=1e-3, momentum=0.01):
        super().__init__()
        mid_channels = int(out_channels * expand_ratio)
        self.main_conv = ConvBNAct(in_channels, mid_channels, 1, eps=eps, momentum=momentum)
        self.short_conv = ConvBNAct(in_channels, mid_channels, 1, eps=eps, momentum=momentum)
        self.final_conv = ConvBNAct(2 * mid_channels, out_channels, 1, eps=eps, momentum=momentum)
        self.blocks = nn.Sequential(*[
            CSPNeXtBlock(mid_channels, mid_channels, expansion=1.0, add_identity=add_identity, eps=eps, momentum=momentum)
            for _ in range(num_blocks)
        ])
        self.attention = ChannelAttention(2 * mid_channels) if channel_attention else nn.Identity()

    def forward(self, x):
        x_short = self.short_conv(x)
        x_main = self.main_conv(x)
        x_main = self.blocks(x_main)
        x = torch.cat((x_main, x_short), dim=1)
        x = self.attention(x)
        return self.final_conv(x)


class CSPNeXtBackboneFromRTMDet(nn.Module):
    arch_settings = {
        'P5': [
            [64, 128, 3, True, False],
            [128, 256, 6, True, False],
            [256, 512, 6, True, False],
            [512, 1024, 3, False, True],
        ]
    }

    def __init__(self, arch='P5', deepen_factor=1.0, widen_factor=1.0,
                 expand_ratio=0.5, channel_attention=True,
                 out_indices=(2, 3, 4), eps=1e-3, momentum=0.01):
        super().__init__()
        arch_setting = self.arch_settings[arch]
        self.out_indices = out_indices
        c0 = int(arch_setting[0][0] * widen_factor)
        self.stem = nn.Sequential(
            ConvBNAct(3, c0 // 2, 3, stride=2, eps=eps, momentum=momentum),
            ConvBNAct(c0 // 2, c0 // 2, 3, stride=1, eps=eps, momentum=momentum),
            ConvBNAct(c0 // 2, c0, 3, stride=1, eps=eps, momentum=momentum),
        )
        self.layers = ['stem']
        for i, (in_c, out_c, num_blocks, add_identity, use_spp) in enumerate(arch_setting):
            in_c = int(in_c * widen_factor)
            out_c = int(out_c * widen_factor)
            num_blocks = max(round(num_blocks * deepen_factor), 1)
            stage = [ConvBNAct(in_c, out_c, 3, stride=2, eps=eps, momentum=momentum)]
            if use_spp:
                stage.append(SPPBottleneck(out_c, out_c, eps=eps, momentum=momentum))
            stage.append(CSPLayer(
                out_c, out_c,
                expand_ratio=expand_ratio,
                num_blocks=num_blocks,
                add_identity=add_identity,
                channel_attention=channel_attention,
                eps=eps,
                momentum=momentum,
            ))
            self.add_module(f'stage{i + 1}', nn.Sequential(*stage))
            self.layers.append(f'stage{i + 1}')

    def forward(self, x):
        outs = []
        for i, layer_name in enumerate(self.layers):
            layer = getattr(self, layer_name)
            x = layer(x)
            if i in self.out_indices:
                outs.append(x)
        return tuple(outs)



def extract_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if 'state_dict' in ckpt:
            return ckpt['state_dict']
        if 'model' in ckpt and isinstance(ckpt['model'], dict):
            return ckpt['model']
    return ckpt



def load_rtmdet_pretrained_backbone(backbone: nn.Module, ckpt_path: Path):
    if not Path(ckpt_path).exists():
        raise FileNotFoundError(ckpt_path)
    ckpt = torch.load(ckpt_path, map_location='cpu')
    state_dict = extract_state_dict(ckpt)

    def remap_key(k: str):
        if not k.startswith('backbone.'):
            return None
        k = k[len('backbone.'):]
        k = k.replace('.conv2.depthwise_conv.', '.conv2.0.')
        k = k.replace('.conv2.pointwise_conv.', '.conv2.1.')
        return k

    filtered = {}
    remapped = 0
    for k, v in state_dict.items():
        new_k = remap_key(k)
        if new_k is None:
            continue
        if new_k != k[len('backbone.'):]:
            remapped += 1
        filtered[new_k] = v

    missing, unexpected = backbone.load_state_dict(filtered, strict=False)
    loaded_keys = set(filtered.keys()) - set(unexpected)
    summary = {
        'checkpoint': str(ckpt_path),
        'raw_keys': len(state_dict),
        'backbone_candidate_keys': len(filtered),
        'remapped_keys': remapped,
        'loaded_keys': len(loaded_keys),
        'missing_keys': len(missing),
        'unexpected_keys': len(unexpected),
    }
    print(json.dumps(summary, indent=2))
    if len(missing):
        print('sample missing:', missing[:20])
    if len(unexpected):
        print('sample unexpected:', unexpected[:20])
    return summary


class _RTMDetMultiScaleBackbone(nn.Module):
    def __init__(self,
                 pretrained_backbone_path: str,
                 arch='P5',
                 deepen_factor=1.0,
                 widen_factor=1.0,
                 expand_ratio=0.5,
                 channel_attention=True,
                 out_indices=(2, 3, 4),
                 fpn_dim: int = 128,
                 groups: int = 8,
                 eps=1e-3,
                 momentum=0.01):
        super().__init__()
        self.fpn_dim = fpn_dim
        self.backbone = CSPNeXtBackboneFromRTMDet(
            arch=arch,
            deepen_factor=deepen_factor,
            widen_factor=widen_factor,
            expand_ratio=expand_ratio,
            channel_attention=channel_attention,
            out_indices=out_indices,
            eps=eps,
            momentum=momentum,
        )
        self.backbone_load_summary = load_rtmdet_pretrained_backbone(self.backbone, Path(pretrained_backbone_path))
        self.laterals = nn.ModuleList([
            nn.Conv2d(256, fpn_dim, 1),
            nn.Conv2d(512, fpn_dim, 1),
            nn.Conv2d(1024, fpn_dim, 1),
        ])
        self.smooth16 = ConvGNAct(fpn_dim, fpn_dim, k=3, s=1, p=1, groups=groups, act=True)
        self.smooth8 = ConvGNAct(fpn_dim, fpn_dim, k=3, s=1, p=1, groups=groups, act=True)
        self.out_proj = nn.Sequential(
            ConvGNAct(fpn_dim * 3, fpn_dim, k=1, s=1, p=0, groups=groups, act=True),
            ConvGNAct(fpn_dim, fpn_dim, k=3, s=1, p=1, groups=groups, act=True),
        )

    def freeze_all_stages(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_last_stages(self, n_last_stages=2):
        self.freeze_all_stages()
        stage_names = ['stage1', 'stage2', 'stage3', 'stage4']
        for name in stage_names[-n_last_stages:]:
            stage = getattr(self.backbone, name)
            for p in stage.parameters():
                p.requires_grad = True

    def forward(self, x):
        feat_s8, feat_s16, feat_s32 = self.backbone(x)
        lat8 = self.laterals[0](feat_s8)
        lat16 = self.laterals[1](feat_s16)
        p32 = self.laterals[2](feat_s32)
        p16 = self.smooth16(lat16 + F.interpolate(p32, size=lat16.shape[-2:], mode='bilinear', align_corners=False))
        p8 = self.smooth8(lat8 + F.interpolate(p16, size=lat8.shape[-2:], mode='bilinear', align_corners=False))
        p16_up = F.interpolate(p16, size=p8.shape[-2:], mode='bilinear', align_corners=False)
        p32_up = F.interpolate(p32, size=p8.shape[-2:], mode='bilinear', align_corners=False)
        fused = self.out_proj(torch.cat([p8, p16_up, p32_up], dim=1))
        return {
            'feat_s8': feat_s8,
            'feat_s16': feat_s16,
            'feat_s32': feat_s32,
            'p8': p8,
            'p16': p16,
            'p32': p32,
            'fused': fused,
        }


class LSSViewTransform2D(nn.Module):
    def __init__(self,
                 in_c: int,
                 context_c: int,
                 depth_bins: int,
                 depth_min: float,
                 depth_max: float,
                 bev_h: int,
                 bev_w: int,
                 bev_res: float,
                 x_range,
                 y_range,
                 z_min: float,
                 z_max: float,
                 groups: int = 8):
        super().__init__()
        self.context_c = context_c
        self.depth_bins = depth_bins
        self.depth_min = float(depth_min)
        self.depth_max = float(depth_max)
        self.bev_h = bev_h
        self.bev_w = bev_w
        self.bev_res = float(bev_res)
        self.x_range = x_range
        self.y_range = y_range
        self.z_min = float(z_min)
        self.z_max = float(z_max)

        self.depth_head = nn.Sequential(
            ConvGNAct(in_c, in_c, k=3, s=1, p=1, groups=groups, act=True),
            nn.Conv2d(in_c, depth_bins, 1),
        )
        self.context_head = nn.Sequential(
            ConvGNAct(in_c, in_c, k=3, s=1, p=1, groups=groups, act=True),
            nn.Conv2d(in_c, context_c, 1),
        )

    def _build_frustum(self, Hf: int, Wf: int, Hi: int, Wi: int, device, dtype):
        depths = torch.linspace(self.depth_min, self.depth_max, self.depth_bins, device=device, dtype=dtype)
        xs = (torch.arange(Wf, device=device, dtype=dtype) + 0.5) * (Wi / Wf)
        ys = (torch.arange(Hf, device=device, dtype=dtype) + 0.5) * (Hi / Hf)
        d, y, x = torch.meshgrid(depths, ys, xs, indexing='ij')
        return x, y, d

    def forward(self, feat_2d: torch.Tensor, intrinsics: torch.Tensor, car2cams: torch.Tensor, image_hw):
        B, N, C, Hf, Wf = feat_2d.shape
        Hi, Wi = image_hw

        feat_bn = feat_2d.reshape(B * N, C, Hf, Wf)
        depth_logits = self.depth_head(feat_bn).float()
        context = self.context_head(feat_bn).float()

        depth_prob = torch.softmax(depth_logits, dim=1)
        depth_prob = depth_prob.reshape(B, N, self.depth_bins, Hf, Wf)
        context = context.reshape(B, N, self.context_c, Hf, Wf)

        x_img, y_img, depth_vals = self._build_frustum(Hf, Wf, Hi, Wi, feat_2d.device, torch.float32)
        x_img = x_img.view(1, 1, self.depth_bins, Hf, Wf)
        y_img = y_img.view(1, 1, self.depth_bins, Hf, Wf)
        depth_vals = depth_vals.view(1, 1, self.depth_bins, Hf, Wf)

        intrinsics = intrinsics.float()
        car2cams = car2cams.float()
        cam2cars = torch.inverse(car2cams.reshape(B * N, 4, 4)).reshape(B, N, 4, 4)

        fx = intrinsics[..., 0, 0].view(B, N, 1, 1, 1)
        fy = intrinsics[..., 1, 1].view(B, N, 1, 1, 1)
        cx = intrinsics[..., 0, 2].view(B, N, 1, 1, 1)
        cy = intrinsics[..., 1, 2].view(B, N, 1, 1, 1)

        X = (x_img - cx) / fx * depth_vals
        Y = (y_img - cy) / fy * depth_vals
        Z = depth_vals.expand(B, N, -1, -1, -1)
        ones = torch.ones_like(Z)
        pts_cam = torch.stack([X, Y, Z, ones], dim=-1)
        pts_car = torch.einsum('bnij,bndhwj->bndhwi', cam2cars, pts_cam)

        world_x = pts_car[..., 0]
        world_y = pts_car[..., 1]
        world_z = pts_car[..., 2]

        x_idx = torch.floor((world_x - self.x_range[0]) / self.bev_res).long()
        y_idx = torch.floor((world_y - self.y_range[0]) / self.bev_res).long()
        valid = (
            (x_idx >= 0) & (x_idx < self.bev_h) &
            (y_idx >= 0) & (y_idx < self.bev_w) &
            (world_z >= self.z_min) & (world_z <= self.z_max)
        )
        linear_idx = x_idx * self.bev_w + y_idx

        feat_vol = context.unsqueeze(3) * depth_prob.unsqueeze(2)
        bev = feat_2d.new_zeros(B, self.context_c, self.bev_h * self.bev_w, dtype=torch.float32)
        counts = feat_2d.new_zeros(B, 1, self.bev_h * self.bev_w, dtype=torch.float32)

        for b in range(B):
            idx_b = linear_idx[b].reshape(-1)
            valid_b = valid[b].reshape(-1)
            if not valid_b.any():
                continue
            feat_b = feat_vol[b].permute(1, 0, 2, 3, 4).reshape(self.context_c, -1)
            idx_valid = idx_b[valid_b]
            feat_valid = feat_b[:, valid_b]
            bev[b].scatter_add_(1, idx_valid.unsqueeze(0).expand(self.context_c, -1), feat_valid)
            counts[b].scatter_add_(1, idx_valid.unsqueeze(0), torch.ones(1, idx_valid.numel(), device=feat_2d.device, dtype=torch.float32))

        bev = bev / counts.clamp(min=1.0)
        bev = bev.reshape(B, self.context_c, self.bev_h, self.bev_w)
        bev = torch.nan_to_num(bev, nan=0.0, posinf=0.0, neginf=0.0)

        debug = {
            'depth_logits': depth_logits.reshape(B, N, self.depth_bins, Hf, Wf),
            'depth_prob': depth_prob,
            'context': context,
            'bev': bev,
            'valid_ratio': valid.float().mean().item(),
        }
        return bev, debug


class StrongBEVEncoderDecoder(nn.Module):
    def __init__(self, in_c: int, base_c: int = 96, groups: int = 8):
        super().__init__()
        self.stem = nn.Sequential(
            ConvGNAct(in_c, base_c, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c, base_c, stride=1, groups=groups),
        )
        self.down1 = nn.Sequential(
            ResidualBlock2d(base_c, base_c * 2, stride=2, groups=groups),
            ResidualBlock2d(base_c * 2, base_c * 2, stride=1, groups=groups),
        )
        self.down2 = nn.Sequential(
            ResidualBlock2d(base_c * 2, base_c * 4, stride=2, groups=groups),
            ResidualBlock2d(base_c * 4, base_c * 4, stride=1, groups=groups),
        )
        self.aspp = ASPP2d(base_c * 4, base_c * 4, rates=(1, 3, 6), groups=groups)
        self.up1 = nn.Sequential(
            ConvGNAct(base_c * 4 + base_c * 2, base_c * 2, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c * 2, base_c * 2, stride=1, groups=groups),
        )
        self.up0 = nn.Sequential(
            ConvGNAct(base_c * 2 + base_c, base_c, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c, base_c, stride=1, groups=groups),
        )
        self.head = nn.Conv2d(base_c, 1, 1)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        b = self.aspp(s2)
        u1 = self.up1(torch.cat([F.interpolate(b, size=s1.shape[-2:], mode='bilinear', align_corners=False), s1], dim=1))
        u0 = self.up0(torch.cat([F.interpolate(u1, size=s0.shape[-2:], mode='bilinear', align_corners=False), s0], dim=1))
        return self.head(u0)


class MultiCamBEVv7RTMDetCSPNeXtLSSClean(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 pretrained_backbone_path: str = './rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth',
                 csp_arch: str = 'P5',
                 csp_deepen_factor: float = 1.0,
                 csp_widen_factor: float = 1.0,
                 csp_expand_ratio: float = 0.5,
                 csp_channel_attention: bool = True,
                 csp_out_indices=(2, 3, 4),
                 fpn_dim: int = 128,
                 context_dim: int = 80,
                 depth_bins: int = 24,
                 depth_min: float = 1.0,
                 depth_max: float = 80.0,
                 world_z_min: float = -2.0,
                 world_z_max: float = 4.5,
                 bev_base_channels: int = 96,
                 bev_gn_groups: int = 8):
        super().__init__()
        self.n_cameras = n_cameras
        self.rover_cond_dim = rover_cond_dim

        self.backbone = _RTMDetMultiScaleBackbone(
            pretrained_backbone_path=pretrained_backbone_path,
            arch=csp_arch,
            deepen_factor=csp_deepen_factor,
            widen_factor=csp_widen_factor,
            expand_ratio=csp_expand_ratio,
            channel_attention=csp_channel_attention,
            out_indices=csp_out_indices,
            fpn_dim=fpn_dim,
            groups=bev_gn_groups,
        )
        if freeze_backbone:
            self.backbone.freeze_all_stages()

        self.view_transform = LSSViewTransform2D(
            in_c=fpn_dim,
            context_c=context_dim,
            depth_bins=depth_bins,
            depth_min=depth_min,
            depth_max=depth_max,
            bev_h=BEV_H,
            bev_w=BEV_W,
            bev_res=BEV_RES,
            x_range=X_RANGE,
            y_range=Y_RANGE,
            z_min=world_z_min,
            z_max=world_z_max,
            groups=bev_gn_groups,
        )

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        self.bev_decoder = StrongBEVEncoderDecoder(
            in_c=context_dim + rover_cond_dim,
            base_c=bev_base_channels,
            groups=bev_gn_groups,
        )

    def forward_debug(self, images, intrinsics, car2cams, rover_ids):
        B, N, C, H, W = images.shape
        assert N == self.n_cameras
        x = images.reshape(B * N, C, H, W)
        back = self.backbone(x)
        feat_s8 = back['feat_s8'].reshape(B, N, back['feat_s8'].shape[1], back['feat_s8'].shape[2], back['feat_s8'].shape[3])
        feat_s16 = back['feat_s16'].reshape(B, N, back['feat_s16'].shape[1], back['feat_s16'].shape[2], back['feat_s16'].shape[3])
        feat_s32 = back['feat_s32'].reshape(B, N, back['feat_s32'].shape[1], back['feat_s32'].shape[2], back['feat_s32'].shape[3])
        fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])
        bev, vt_debug = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, BEV_H, BEV_W)
        logits = self.bev_decoder(torch.cat([bev, rover_map], dim=1))
        return {
            'feat_s8': feat_s8,
            'feat_s16': feat_s16,
            'feat_s32': feat_s32,
            'image_fused': fused,
            'depth_logits': vt_debug['depth_logits'],
            'depth_prob': vt_debug['depth_prob'],
            'bev_raw': vt_debug['bev'],
            'valid_ratio': vt_debug['valid_ratio'],
            'logits': logits,
        }

    def forward(self, images, intrinsics, car2cams, rover_ids):
        dbg = self.forward_debug(images, intrinsics, car2cams, rover_ids)
        logits = torch.nan_to_num(dbg['logits'], nan=0.0, posinf=0.0, neginf=0.0)
        return logits



def load_resume_state(core_model, ema_model, optimizer, scheduler, scaler, run_dir: Path):
    resume_ckpt = Path(cfg.get('resume_ckpt', ''))
    if not cfg.get('resume_training', False) or not resume_ckpt.exists():
        return {
            'enabled': False,
            'start_epoch': 0,
            'best_iou': -1.0,
            'best_ema_iou': -1.0,
            'log': [],
            'elapsed_minutes': 0.0,
        }

    ckpt = torch.load(resume_ckpt, map_location='cpu')
    core_model.load_state_dict(ckpt['model'], strict=False)
    if 'ema' in ckpt:
        ema_model.load_state_dict(ckpt['ema'], strict=False)
    if 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    if 'scheduler' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler'])
    if 'scaler' in ckpt and ckpt['scaler'] is not None:
        scaler.load_state_dict(ckpt['scaler'])

    start_epoch = int(ckpt.get('epoch', -1)) + 1
    log_path = run_dir / 'log.csv'
    log_rows = []
    elapsed_minutes = 0.0
    if log_path.exists():
        log_rows = pd.read_csv(log_path).to_dict('records')
        if len(log_rows):
            elapsed_minutes = float(log_rows[-1].get('minutes', 0.0) or 0.0)

    best_iou = float(ckpt.get('best_iou', -1.0))
    best_ema_iou = float(ckpt.get('best_ema_iou', -1.0))
    best_path = run_dir / 'best.pt'
    ema_best_path = run_dir / 'ema_best.pt'
    if best_path.exists():
        try:
            best_iou = max(best_iou, float(torch.load(best_path, map_location='cpu').get('best_iou', -1.0)))
        except Exception:
            pass
    if ema_best_path.exists():
        try:
            best_ema_iou = max(best_ema_iou, float(torch.load(ema_best_path, map_location='cpu').get('best_ema_iou', -1.0)))
        except Exception:
            pass

    print('resumed from', resume_ckpt)
    print('  start_epoch:', start_epoch)
    print('  best_iou so far:', best_iou)
    print('  best_ema_iou so far:', best_ema_iou)
    print('  prior log rows:', len(log_rows))
    return {
        'enabled': True,
        'start_epoch': start_epoch,
        'best_iou': best_iou,
        'best_ema_iou': best_ema_iou,
        'log': log_rows,
        'elapsed_minutes': elapsed_minutes,
    }


def gn_groups(channels: int, requested: int = 8) -> int:
    g = min(requested, channels)
    while channels % g != 0 and g > 1:
        g -= 1
    return max(g, 1)


class ConvGNAct(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1, groups=8, act=True):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, k, stride=s, padding=p, bias=False), nn.GroupNorm(gn_groups(out_c, groups), out_c)]
        if act:
            layers.append(nn.SiLU(inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)


class ResidualBlock2d(nn.Module):
    def __init__(self, in_c, out_c, stride=1, groups=8):
        super().__init__()
        self.conv1 = ConvGNAct(in_c, out_c, k=3, s=stride, p=1, groups=groups, act=True)
        self.conv2 = ConvGNAct(out_c, out_c, k=3, s=1, p=1, groups=groups, act=False)
        self.skip = ConvGNAct(in_c, out_c, k=1, s=stride, p=0, groups=groups, act=False) if (stride != 1 or in_c != out_c) else nn.Identity()
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        return self.act(self.conv2(self.conv1(x)) + self.skip(x))


class ASPP2d(nn.Module):
    def __init__(self, in_c, out_c, rates=(1, 3, 6), groups=8):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=r, dilation=r, bias=False),
                nn.GroupNorm(gn_groups(out_c, groups), out_c),
                nn.SiLU(inplace=True),
            )
            for r in rates
        ])
        self.proj = ConvGNAct(out_c * len(rates), out_c, k=1, s=1, p=0, groups=groups, act=True)
    def forward(self, x):
        return self.proj(torch.cat([b(x) for b in self.branches], dim=1))


class _DINOv2MultiScaleBackbone(nn.Module):
    def __init__(self, hub_repo='facebookresearch/dinov2', backbone_name='dinov2_vitb14', out_dim=768, patch_size=14, tap_layers=(2,5,8,11), neck_dim=128, groups=8):
        super().__init__()
        self.hub_repo = hub_repo
        self.backbone_name = backbone_name
        self.out_dim = out_dim
        self.patch_size = patch_size
        self.tap_layers = tuple(tap_layers)
        self.vit = self._load_hub_model()
        self.laterals = nn.ModuleList([nn.Conv2d(out_dim, neck_dim, 1) for _ in self.tap_layers])
        self.fuse = nn.Sequential(
            ConvGNAct(len(self.tap_layers) * neck_dim, neck_dim, k=3, s=1, p=1, groups=groups, act=True),
            ConvGNAct(neck_dim, neck_dim, k=3, s=1, p=1, groups=groups, act=True),
        )
        self.down1 = ConvGNAct(neck_dim, neck_dim, k=3, s=2, p=1, groups=groups, act=True)
        self.down2 = ConvGNAct(neck_dim, neck_dim, k=3, s=2, p=1, groups=groups, act=True)
        self.neck_out = ConvGNAct(neck_dim * 3, neck_dim, k=1, s=1, p=0, groups=groups, act=True)

    def _load_hub_model(self):
        last_err = None
        attempts = [
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, pretrained=True),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, source='github'),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, source='github', pretrained=True),
        ]
        for kwargs in attempts:
            try:
                return torch.hub.load(**kwargs)
            except Exception as e:
                last_err = e
        raise RuntimeError(f'Failed to load DINOv2 backbone. Last error: {last_err}')

    def _reshape_tokens(self, tokens, H, W):
        B = tokens.shape[0]
        expected_tokens = (H // self.patch_size) * (W // self.patch_size)
        if tokens.shape[1] == expected_tokens + 1:
            tokens = tokens[:, 1:, :]
        Hp = H // self.patch_size
        Wp = W // self.patch_size
        return tokens.transpose(1, 2).reshape(B, self.out_dim, Hp, Wp).contiguous()

    def _extract_intermediate(self, x):
        H, W = x.shape[-2:]
        try:
            feats = self.vit.get_intermediate_layers(x, n=list(self.tap_layers), reshape=True, return_class_token=False)
        except Exception:
            feats = self.vit.get_intermediate_layers(x, n=len(self.tap_layers), reshape=True, return_class_token=False)
        out = []
        for feat in feats:
            if isinstance(feat, (tuple, list)):
                feat = feat[0]
            if feat.ndim == 3:
                feat = self._reshape_tokens(feat, H, W)
            out.append(feat)
        return out

    def forward(self, x):
        feats = self._extract_intermediate(x)
        laterals = [proj(feat) for proj, feat in zip(self.laterals, feats)]
        p0 = self.fuse(torch.cat(laterals, dim=1))
        p1 = self.down1(p0)
        p2 = self.down2(p1)
        p1_up = F.interpolate(p1, size=p0.shape[-2:], mode='bilinear', align_corners=False)
        p2_up = F.interpolate(p2, size=p0.shape[-2:], mode='bilinear', align_corners=False)
        fused = self.neck_out(torch.cat([p0, p1_up, p2_up], dim=1))
        return {'fused': fused}


class LSSViewTransform2D(nn.Module):
    def __init__(self, in_c, context_c, depth_bins, depth_min, depth_max, bev_h, bev_w, bev_res, x_range, y_range, z_min, z_max, groups=8):
        super().__init__()
        self.context_c = context_c
        self.depth_bins = depth_bins
        self.depth_min = float(depth_min)
        self.depth_max = float(depth_max)
        self.bev_h = bev_h
        self.bev_w = bev_w
        self.bev_res = float(bev_res)
        self.x_range = x_range
        self.y_range = y_range
        self.z_min = float(z_min)
        self.z_max = float(z_max)
        self.depth_head = nn.Sequential(ConvGNAct(in_c, in_c, 3, 1, 1, groups, True), nn.Conv2d(in_c, depth_bins, 1))
        self.context_head = nn.Sequential(ConvGNAct(in_c, in_c, 3, 1, 1, groups, True), nn.Conv2d(in_c, context_c, 1))

    def _build_frustum(self, Hf, Wf, Hi, Wi, device, dtype):
        depths = torch.linspace(self.depth_min, self.depth_max, self.depth_bins, device=device, dtype=dtype)
        xs = (torch.arange(Wf, device=device, dtype=dtype) + 0.5) * (Wi / Wf)
        ys = (torch.arange(Hf, device=device, dtype=dtype) + 0.5) * (Hi / Hf)
        d, y, x = torch.meshgrid(depths, ys, xs, indexing='ij')
        return x, y, d

    def forward(self, feat_2d, intrinsics, car2cams, image_hw):
        B, N, C, Hf, Wf = feat_2d.shape
        Hi, Wi = image_hw
        feat_bn = feat_2d.reshape(B * N, C, Hf, Wf)
        depth_logits = self.depth_head(feat_bn).float()
        context = self.context_head(feat_bn).float()
        depth_prob = torch.softmax(depth_logits, dim=1).reshape(B, N, self.depth_bins, Hf, Wf)
        context = context.reshape(B, N, self.context_c, Hf, Wf)

        x_img, y_img, depth_vals = self._build_frustum(Hf, Wf, Hi, Wi, feat_2d.device, torch.float32)
        x_img = x_img.view(1, 1, self.depth_bins, Hf, Wf)
        y_img = y_img.view(1, 1, self.depth_bins, Hf, Wf)
        depth_vals = depth_vals.view(1, 1, self.depth_bins, Hf, Wf)

        intrinsics = intrinsics.float()
        car2cams = car2cams.float()
        cam2cars = torch.inverse(car2cams.reshape(B * N, 4, 4)).reshape(B, N, 4, 4)

        fx = intrinsics[..., 0, 0].view(B, N, 1, 1, 1)
        fy = intrinsics[..., 1, 1].view(B, N, 1, 1, 1)
        cx = intrinsics[..., 0, 2].view(B, N, 1, 1, 1)
        cy = intrinsics[..., 1, 2].view(B, N, 1, 1, 1)

        X = (x_img - cx) / fx * depth_vals
        Y = (y_img - cy) / fy * depth_vals
        Z = depth_vals.expand(B, N, -1, -1, -1)
        ones = torch.ones_like(Z)
        pts_cam = torch.stack([X, Y, Z, ones], dim=-1)
        pts_car = torch.einsum('bnij,bndhwj->bndhwi', cam2cars, pts_cam)

        world_x = pts_car[..., 0]
        world_y = pts_car[..., 1]
        world_z = pts_car[..., 2]
        x_idx = torch.floor((world_x - self.x_range[0]) / self.bev_res).long()
        y_idx = torch.floor((world_y - self.y_range[0]) / self.bev_res).long()
        valid = ((x_idx >= 0) & (x_idx < self.bev_h) & (y_idx >= 0) & (y_idx < self.bev_w) & (world_z >= self.z_min) & (world_z <= self.z_max))
        linear_idx = x_idx * self.bev_w + y_idx

        feat_vol = context.unsqueeze(3) * depth_prob.unsqueeze(2)
        bev = feat_2d.new_zeros(B, self.context_c, self.bev_h * self.bev_w, dtype=torch.float32)
        counts = feat_2d.new_zeros(B, 1, self.bev_h * self.bev_w, dtype=torch.float32)
        for b in range(B):
            idx_b = linear_idx[b].reshape(-1)
            valid_b = valid[b].reshape(-1)
            if not valid_b.any():
                continue
            feat_b = feat_vol[b].permute(1, 0, 2, 3, 4).reshape(self.context_c, -1)
            idx_valid = idx_b[valid_b]
            feat_valid = feat_b[:, valid_b]
            bev[b].scatter_add_(1, idx_valid.unsqueeze(0).expand(self.context_c, -1), feat_valid)
            counts[b].scatter_add_(1, idx_valid.unsqueeze(0), torch.ones(1, idx_valid.numel(), device=feat_2d.device, dtype=torch.float32))
        bev = bev / counts.clamp(min=1.0)
        return torch.nan_to_num(bev.reshape(B, self.context_c, self.bev_h, self.bev_w), nan=0.0, posinf=0.0, neginf=0.0)


class StrongBEVEncoderDecoder(nn.Module):
    def __init__(self, in_c, base_c=96, groups=8):
        super().__init__()
        self.stem = nn.Sequential(ConvGNAct(in_c, base_c, 3, 1, 1, groups, True), ResidualBlock2d(base_c, base_c, 1, groups))
        self.down1 = nn.Sequential(ResidualBlock2d(base_c, base_c * 2, 2, groups), ResidualBlock2d(base_c * 2, base_c * 2, 1, groups))
        self.down2 = nn.Sequential(ResidualBlock2d(base_c * 2, base_c * 4, 2, groups), ResidualBlock2d(base_c * 4, base_c * 4, 1, groups))
        self.aspp = ASPP2d(base_c * 4, base_c * 4, (1,3,6), groups)
        self.up1 = nn.Sequential(ConvGNAct(base_c * 4 + base_c * 2, base_c * 2, 3, 1, 1, groups, True), ResidualBlock2d(base_c * 2, base_c * 2, 1, groups))
        self.up0 = nn.Sequential(ConvGNAct(base_c * 2 + base_c, base_c, 3, 1, 1, groups, True), ResidualBlock2d(base_c, base_c, 1, groups))
        self.head = nn.Conv2d(base_c, 1, 1)
    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        b = self.aspp(s2)
        u1 = self.up1(torch.cat([F.interpolate(b, size=s1.shape[-2:], mode='bilinear', align_corners=False), s1], dim=1))
        u0 = self.up0(torch.cat([F.interpolate(u1, size=s0.shape[-2:], mode='bilinear', align_corners=False), s0], dim=1))
        return self.head(u0)


class MultiCamBEVv6DINOv2LSSClean(nn.Module):
    def __init__(self, num_rover_classes, rover_emb_dim=8, rover_cond_dim=8, n_cameras=4, freeze_backbone=False, hub_repo='facebookresearch/dinov2', backbone_name='dinov2_vitb14', backbone_out_dim=768, patch_size=14, tap_layers=(2,5,8,11), neck_dim=128, context_dim=80, depth_bins=24, depth_min=1.0, depth_max=80.0, world_z_min=-2.0, world_z_max=4.5, bev_base_channels=96, bev_gn_groups=8):
        super().__init__()
        self.n_cameras = n_cameras
        self.rover_cond_dim = rover_cond_dim
        self.backbone = _DINOv2MultiScaleBackbone(hub_repo, backbone_name, backbone_out_dim, patch_size, tap_layers, neck_dim, bev_gn_groups)
        self.view_transform = LSSViewTransform2D(neck_dim, context_dim, depth_bins, depth_min, depth_max, BEV_H, BEV_W, BEV_RES, X_RANGE, Y_RANGE, world_z_min, world_z_max, bev_gn_groups)
        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        self.rover_mlp = nn.Sequential(nn.Linear(rover_emb_dim, 16), nn.ReLU(inplace=True), nn.Linear(16, rover_cond_dim), nn.ReLU(inplace=True))
        self.bev_decoder = StrongBEVEncoderDecoder(context_dim + rover_cond_dim, bev_base_channels, bev_gn_groups)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C, H, W = images.shape
        x = images.reshape(B * N, C, H, W)
        back = self.backbone(x)
        fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])
        bev = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, BEV_H, BEV_W)
        logits = self.bev_decoder(torch.cat([bev, rover_map], dim=1))
        return torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)


In [7]:
def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def infer_model_type(run_name: str, ckpt: dict):
    model_type = ckpt.get('model_type')
    if model_type:
        return model_type
    if 'v6' in run_name or 'dinov2' in run_name:
        return 'v6_dinov2_lss_bev_cleaned'
    if 'v7' in run_name or 'rtmdet' in run_name:
        return 'v7_rtmdet_cspnext_lss_bev_cleaned'
    raise RuntimeError(f'Cannot infer model_type for run={run_name}')


def normalize_cfg(raw_cfg: dict, model_type: str):
    cfg = dict(raw_cfg or {})
    if model_type.startswith('v6'):
        return {
            'img_hw': tuple(cfg.get('img_hw', [384, 704])),
            'rover_emb_dim': int(cfg.get('rover_emb_dim', 8)),
            'rover_cond_dim': int(cfg.get('rover_cond_dim', 8)),
            'backbone_name': cfg.get('backbone_name', 'dinov2_vitb14'),
            'hub_repo': cfg.get('hub_repo', 'facebookresearch/dinov2'),
            'backbone_out_dim': int(cfg.get('backbone_out_dim', 768)),
            'patch_size': int(cfg.get('patch_size', 14)),
            'tap_layers': tuple(cfg.get('tap_layers', [2, 5, 8, 11])),
            'neck_dim': int(cfg.get('neck_dim', 128)),
            'context_dim': int(cfg.get('context_dim', 80)),
            'depth_bins': int(cfg.get('depth_bins', 24)),
            'depth_min': float(cfg.get('depth_min', 1.0)),
            'depth_max': float(cfg.get('depth_max', 80.0)),
            'world_z_min': float(cfg.get('world_z_min', -2.0)),
            'world_z_max': float(cfg.get('world_z_max', 4.5)),
            'bev_base_channels': int(cfg.get('bev_base_channels', 96)),
            'bev_gn_groups': int(cfg.get('bev_gn_groups', 8)),
            'use_amp': bool(cfg.get('use_amp', True)),
            'min_rover_count': int(cfg.get('min_rover_count', 30)),
            'topk_rovers': int(cfg.get('topk_rovers', 25)),
        }
    return {
        'img_hw': tuple(cfg.get('img_hw', [384, 704])),
        'rover_emb_dim': int(cfg.get('rover_emb_dim', 8)),
        'rover_cond_dim': int(cfg.get('rover_cond_dim', 8)),
        'rtmdet_backbone_ckpt': cfg.get('rtmdet_backbone_ckpt', str(DEFAULT_RTMDET_CKPT)),
        'csp_arch': cfg.get('csp_arch', 'P5'),
        'csp_deepen_factor': float(cfg.get('csp_deepen_factor', 1.0)),
        'csp_widen_factor': float(cfg.get('csp_widen_factor', 1.0)),
        'csp_expand_ratio': float(cfg.get('csp_expand_ratio', 0.5)),
        'csp_channel_attention': bool(cfg.get('csp_channel_attention', True)),
        'csp_out_indices': tuple(cfg.get('csp_out_indices', [2, 3, 4])),
        'fpn_dim': int(cfg.get('fpn_dim', 128)),
        'context_dim': int(cfg.get('context_dim', 80)),
        'depth_bins': int(cfg.get('depth_bins', 24)),
        'depth_min': float(cfg.get('depth_min', 1.0)),
        'depth_max': float(cfg.get('depth_max', 80.0)),
        'world_z_min': float(cfg.get('world_z_min', -2.0)),
        'world_z_max': float(cfg.get('world_z_max', 4.5)),
        'bev_base_channels': int(cfg.get('bev_base_channels', 96)),
        'bev_gn_groups': int(cfg.get('bev_gn_groups', 8)),
        'use_amp': bool(cfg.get('use_amp', True)),
        'min_rover_count': int(cfg.get('min_rover_count', 30)),
        'topk_rovers': int(cfg.get('topk_rovers', 25)),
    }


def build_model_for_ckpt(model_type: str, cfg: dict, rover_vocab: dict):
    if model_type.startswith('v6'):
        return MultiCamBEVv6DINOv2LSSClean(
            num_rover_classes=len(rover_vocab),
            rover_emb_dim=cfg['rover_emb_dim'],
            rover_cond_dim=cfg['rover_cond_dim'],
            freeze_backbone=False,
            hub_repo=cfg['hub_repo'],
            backbone_name=cfg['backbone_name'],
            backbone_out_dim=cfg['backbone_out_dim'],
            patch_size=cfg['patch_size'],
            tap_layers=cfg['tap_layers'],
            neck_dim=cfg['neck_dim'],
            context_dim=cfg['context_dim'],
            depth_bins=cfg['depth_bins'],
            depth_min=cfg['depth_min'],
            depth_max=cfg['depth_max'],
            world_z_min=cfg['world_z_min'],
            world_z_max=cfg['world_z_max'],
            bev_base_channels=cfg['bev_base_channels'],
            bev_gn_groups=cfg['bev_gn_groups'],
        ).to(device)
    return MultiCamBEVv7RTMDetCSPNeXtLSSClean(
        num_rover_classes=len(rover_vocab),
        rover_emb_dim=cfg['rover_emb_dim'],
        rover_cond_dim=cfg['rover_cond_dim'],
        freeze_backbone=False,
        pretrained_backbone_path=str(Path(cfg['rtmdet_backbone_ckpt']) if Path(cfg['rtmdet_backbone_ckpt']).exists() else DEFAULT_RTMDET_CKPT),
        csp_arch=cfg['csp_arch'],
        csp_deepen_factor=cfg['csp_deepen_factor'],
        csp_widen_factor=cfg['csp_widen_factor'],
        csp_expand_ratio=cfg['csp_expand_ratio'],
        csp_channel_attention=cfg['csp_channel_attention'],
        csp_out_indices=cfg['csp_out_indices'],
        fpn_dim=cfg['fpn_dim'],
        context_dim=cfg['context_dim'],
        depth_bins=cfg['depth_bins'],
        depth_min=cfg['depth_min'],
        depth_max=cfg['depth_max'],
        world_z_min=cfg['world_z_min'],
        world_z_max=cfg['world_z_max'],
        bev_base_channels=cfg['bev_base_channels'],
        bev_gn_groups=cfg['bev_gn_groups'],
    ).to(device)


def get_candidate_rover_vocab(ckpt: dict, run_cfg: dict):
    rover_vocab = ckpt.get('rover_vocab')
    if rover_vocab is not None:
        return rover_vocab
    return build_rover_vocab_from_train(train_info, min_count=run_cfg['min_rover_count'], topk=run_cfg['topk_rovers'])


def build_loaders(img_hw, rover_vocab, val_batch_size=1, test_batch_size=1):
    ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=img_hw, rover_vocab=rover_vocab)
    ds_test = BEVDatasetV4Clean(test_info, mode='test', img_hw=img_hw, rover_vocab=rover_vocab)
    loader_val = DataLoader(ds_val, batch_size=val_batch_size, shuffle=False, num_workers=0, pin_memory=(device.type == 'cuda'))
    loader_test = DataLoader(ds_test, batch_size=test_batch_size, shuffle=False, num_workers=2, pin_memory=(device.type == 'cuda'))
    return loader_val, loader_test


def load_candidate(run_spec: dict, ckpt_kind: str):
    ckpt_path = run_spec['run_dir'] / f'{ckpt_kind}.pt'
    assert ckpt_path.exists(), ckpt_path
    ckpt = torch.load(ckpt_path, map_location='cpu')
    model_type = infer_model_type(run_spec['name'], ckpt)
    raw_cfg = ckpt.get('cfg')
    if raw_cfg is None:
        config_json = run_spec['run_dir'] / 'config.json'
        raw_cfg = json.loads(config_json.read_text()) if config_json.exists() else {}
    run_cfg = normalize_cfg(raw_cfg, model_type)
    rover_vocab = get_candidate_rover_vocab(ckpt, run_cfg)
    model = build_model_for_ckpt(model_type, run_cfg, rover_vocab)
    use_ema = (ckpt_kind == 'ema_best') and ('ema' in ckpt)
    state = ckpt['ema'] if use_ema else ckpt['model']
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"loaded {run_spec['name']}::{ckpt_kind} | model_type={model_type} | missing={len(missing)} unexpected={len(unexpected)}")
    loader_val, loader_test = build_loaders(run_cfg['img_hw'], rover_vocab)
    return {
        'run_name': run_spec['name'],
        'run_dir': run_spec['run_dir'],
        'ckpt_kind': ckpt_kind,
        'ckpt_path': ckpt_path,
        'ckpt': ckpt,
        'model_type': model_type,
        'cfg': run_cfg,
        'rover_vocab': rover_vocab,
        'model': model,
        'loader_val': loader_val,
        'loader_test': loader_test,
        'use_amp': run_cfg['use_amp'],
    }


In [8]:
@torch.inference_mode()
def collect_val_probs(model, loader, cache_path: Path):
    if cache_path.exists():
        d = torch.load(cache_path, map_location='cpu')
        return d['probs'], d['gt']
    model.eval()
    probs_list, gt_list = [], []
    for batch in tqdm(loader, desc=f'collect val probs -> {cache_path.name}'):
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        gt = batch['gt'].cpu()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        probs = torch.sigmoid(logits.float()).cpu().to(torch.float16)
        probs_list.append(probs)
        gt_list.append(gt)
    probs = torch.cat(probs_list, dim=0)
    gt = torch.cat(gt_list, dim=0)
    torch.save({'probs': probs, 'gt': gt}, cache_path)
    return probs, gt


def threshold_sweep_from_cached_probs(probs, gt, thresholds):
    inter = torch.zeros(len(thresholds), dtype=torch.float64)
    union = torch.zeros(len(thresholds), dtype=torch.float64)
    valid = gt != 255
    gt_b = ((gt == 1) & valid).float()
    for i, t in enumerate(thresholds):
        pred = ((probs > t) & valid).float()
        inter[i] = (pred * gt_b).sum().item()
        union[i] = ((pred + gt_b).clamp(0, 1)).sum().item()
    return {float(t): float(inter[i] / max(union[i].item(), 1.0)) for i, t in enumerate(thresholds)}


@torch.inference_mode()
def collect_test_probs(model, loader, cache_path: Path):
    if cache_path.exists():
        return torch.load(cache_path, map_location='cpu')
    model.eval()
    probs_list = []
    for batch in tqdm(loader, desc=f'collect test probs -> {cache_path.name}'):
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        probs = torch.sigmoid(logits.float()).cpu().to(torch.float16)
        probs_list.append(probs)
    probs = torch.cat(probs_list, dim=0)
    torch.save(probs, cache_path)
    return probs


def candidate_slug(candidate: dict) -> str:
    return f"{candidate['run_name']}__{candidate['ckpt_kind']}"


def save_json(obj, path: Path):
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False))


def _pred_name_from_row(row):
    if 'predicted_occupancy_grid' in row:
        return Path(row['predicted_occupancy_grid']).name
    if 'predicted_static_grid' in row:
        return Path(row['predicted_static_grid']).name
    return f'{int(row.name)}.npy'


def make_submission_from_probs(test_probs: torch.Tensor, threshold: float, out_root: Path, tag: str):
    pred_dir = out_root / f'predicted_static_grids_{tag}_t_{threshold:.3f}'.replace('.', 'p')
    pred_dir.mkdir(parents=True, exist_ok=True)
    for p in pred_dir.glob('*.npy'):
        p.unlink()
    preds = (test_probs > threshold).numpy().astype(np.int32)
    assert len(preds) == len(test_info)
    for i, row in test_info.iterrows():
        out_name = _pred_name_from_row(row)
        np.save(pred_dir / out_name, preds[i].reshape(1, BEV_H, BEV_W))
    zip_path = out_root / f'submission_{tag}_t_{threshold:.3f}.zip'.replace('.', 'p')
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        zf.write(DATA_TEST / 'info.csv', arcname='info.csv')
        for npy in sorted(pred_dir.glob('*.npy')):
            zf.write(npy, arcname=f'predicted_static_grids/{npy.name}')
    h = hashlib.sha256()
    with open(zip_path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return {'zip_path': str(zip_path), 'size_mb': round(zip_path.stat().st_size / 1e6, 3), 'sha256': h.hexdigest()}


In [10]:
@torch.inference_mode()
def collect_val_probs(model, loader, cache_path: Path, use_amp: bool = True):
    if cache_path.exists():
        d = torch.load(cache_path, map_location='cpu')
        return d['probs'], d['gt']

    model.eval()
    probs_list, gt_list = [], []

    for batch in tqdm(loader, desc=f'collect val probs -> {cache_path.name}'):
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        gt = batch['gt'].cpu()

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and use_amp)):
            logits = model(images, intr, c2c, rover_id)

        probs = torch.sigmoid(logits.float()).cpu().to(torch.float16)
        probs_list.append(probs)
        gt_list.append(gt)

    probs = torch.cat(probs_list, dim=0)
    gt = torch.cat(gt_list, dim=0)
    torch.save({'probs': probs, 'gt': gt}, cache_path)
    return probs, gt


@torch.inference_mode()
def collect_test_probs(model, loader, cache_path: Path, use_amp: bool = True):
    if cache_path.exists():
        return torch.load(cache_path, map_location='cpu')

    model.eval()
    probs_list = []

    for batch in tqdm(loader, desc=f'collect test probs -> {cache_path.name}'):
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and use_amp)):
            logits = model(images, intr, c2c, rover_id)

        probs = torch.sigmoid(logits.float()).cpu().to(torch.float16)
        probs_list.append(probs)

    probs = torch.cat(probs_list, dim=0)
    torch.save(probs, cache_path)
    return probs


In [11]:
manifest_rows = []

for run_spec in RUN_SPECS:
    for ckpt_kind in ['best', 'ema_best']:
        cleanup_cuda()
        candidate = load_candidate(run_spec, ckpt_kind)
        slug = candidate_slug(candidate)
        candidate_dir = OUT_DIR / slug
        candidate_dir.mkdir(parents=True, exist_ok=True)

        val_probs, val_gt = collect_val_probs(
            candidate['model'],
            candidate['loader_val'],
            candidate_dir / 'val_probs.pt',
            use_amp=candidate['use_amp'],
        )

        test_probs = collect_test_probs(
            candidate['model'],
            candidate['loader_test'],
            candidate_dir / 'test_probs.pt',
            use_amp=candidate['use_amp'],
        )


        meta = {
            'slug': slug,
            'run_name': candidate['run_name'],
            'ckpt_kind': candidate['ckpt_kind'],
            'model_type': candidate['model_type'],
            'ckpt_path': str(candidate['ckpt_path']),
            'img_hw': list(candidate['cfg']['img_hw']),
            'num_rover_classes': len(candidate['rover_vocab']),
            'val_probs_path': str(candidate_dir / 'val_probs.pt'),
            'test_probs_path': str(candidate_dir / 'test_probs.pt'),
            'val_shape': list(val_probs.shape),
            'test_shape': list(test_probs.shape),
        }
        save_json(meta, candidate_dir / 'meta.json')
        manifest_rows.append(meta)

        print('=' * 100)
        print(slug)
        print('val_probs:', tuple(val_probs.shape), '| test_probs:', tuple(test_probs.shape))

        del candidate['model']
        cleanup_cuda()

manifest_df = pd.DataFrame(manifest_rows).sort_values(['run_name', 'ckpt_kind']).reset_index(drop=True)
manifest_df.to_csv(OUT_DIR / 'manifest.csv', index=False)
save_json(manifest_rows, OUT_DIR / 'manifest.json')
display(manifest_df)


Using cache found in /tmp/xdg_cache/torch/hub/facebookresearch_dinov2_main


loaded v6_dino::best | model_type=v6_dinov2_lss_bev_cleaned | missing=0 unexpected=0


collect test probs -> test_probs.pt: 100%|██████████| 2000/2000 [04:22<00:00,  7.62it/s]


v6_dino__best
val_probs: (220, 1, 188, 126) | test_probs: (2000, 1, 188, 126)


Using cache found in /tmp/xdg_cache/torch/hub/facebookresearch_dinov2_main


loaded v6_dino::ema_best | model_type=v6_dinov2_lss_bev_cleaned | missing=0 unexpected=0


collect test probs -> test_probs.pt: 100%|██████████| 2000/2000 [04:26<00:00,  7.49it/s]


v6_dino__ema_best
val_probs: (220, 1, 188, 126) | test_probs: (2000, 1, 188, 126)
{
  "checkpoint": "rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth",
  "raw_keys": 874,
  "backbone_candidate_keys": 458,
  "remapped_keys": 216,
  "loaded_keys": 458,
  "missing_keys": 0,
  "unexpected_keys": 0
}
loaded v7_rtmdet::best | model_type=v7_rtmdet_cspnext_lss_bev_cleaned | missing=0 unexpected=0


collect val probs -> val_probs.pt:   0%|          | 0/220 [00:00<?, ?it/s]


ValueError: not enough values to unpack (expected 2, got 1)

In [12]:
import shutil
import torch

# Patch v7 model so it tolerates both LSS return styles:
# - tuple: (bev, debug)
# - tensor: bev
# - dict with 'bev'
def _patched_v7_forward_debug(self, images, intrinsics, car2cams, rover_ids):
    B, N, C, H, W = images.shape
    assert N == self.n_cameras

    x = images.reshape(B * N, C, H, W)
    back = self.backbone(x)

    feat_s8 = back['feat_s8'].reshape(B, N, back['feat_s8'].shape[1], back['feat_s8'].shape[2], back['feat_s8'].shape[3])
    feat_s16 = back['feat_s16'].reshape(B, N, back['feat_s16'].shape[1], back['feat_s16'].shape[2], back['feat_s16'].shape[3])
    feat_s32 = back['feat_s32'].reshape(B, N, back['feat_s32'].shape[1], back['feat_s32'].shape[2], back['feat_s32'].shape[3])
    fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])

    vt_out = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
    if isinstance(vt_out, tuple):
        bev, vt_debug = vt_out
    elif isinstance(vt_out, dict):
        bev = vt_out['bev'] if 'bev' in vt_out else vt_out['bev_raw']
        vt_debug = vt_out
    else:
        bev = vt_out
        vt_debug = {'bev': bev, 'depth_prob': None, 'valid_ratio': None}

    rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
    rover_map = rover_feat.expand(-1, -1, BEV_H, BEV_W)
    logits = self.bev_decoder(torch.cat([bev, rover_map], dim=1))

    return {
        'feat_s8': feat_s8,
        'feat_s16': feat_s16,
        'feat_s32': feat_s32,
        'image_fused': fused,
        'depth_prob': vt_debug.get('depth_prob', None) if isinstance(vt_debug, dict) else None,
        'bev_raw': vt_debug.get('bev', bev) if isinstance(vt_debug, dict) else bev,
        'valid_ratio': vt_debug.get('valid_ratio', None) if isinstance(vt_debug, dict) else None,
        'logits': logits,
    }

def _patched_v7_forward(self, images, intrinsics, car2cams, rover_ids):
    dbg = self.forward_debug(images, intrinsics, car2cams, rover_ids)
    return torch.nan_to_num(dbg['logits'], nan=0.0, posinf=0.0, neginf=0.0)

MultiCamBEVv7RTMDetCSPNeXtLSSClean.forward_debug = _patched_v7_forward_debug
MultiCamBEVv7RTMDetCSPNeXtLSSClean.forward = _patched_v7_forward

print("patched MultiCamBEVv7RTMDetCSPNeXtLSSClean")


patched MultiCamBEVv7RTMDetCSPNeXtLSSClean


In [13]:
# Optional: clear old failed outputs for rtmdet only
for slug in ['v7_rtmdet__best', 'v7_rtmdet__ema_best']:
    shutil.rmtree(OUT_DIR / slug, ignore_errors=True)

rtmdet_spec = next(r for r in RUN_SPECS if r['name'] == 'v7_rtmdet')
rtmdet_rows = []

for ckpt_kind in ['best', 'ema_best']:
    cleanup_cuda()

    candidate = load_candidate(rtmdet_spec, ckpt_kind)
    slug = candidate_slug(candidate)
    candidate_dir = OUT_DIR / slug
    candidate_dir.mkdir(parents=True, exist_ok=True)

    val_probs, val_gt = collect_val_probs(
        candidate['model'],
        candidate['loader_val'],
        candidate_dir / 'val_probs.pt',
        use_amp=candidate['use_amp'],
    )

    test_probs = collect_test_probs(
        candidate['model'],
        candidate['loader_test'],
        candidate_dir / 'test_probs.pt',
        use_amp=candidate['use_amp'],
    )

    meta = {
        'slug': slug,
        'run_name': candidate['run_name'],
        'ckpt_kind': candidate['ckpt_kind'],
        'model_type': candidate['model_type'],
        'ckpt_path': str(candidate['ckpt_path']),
        'img_hw': list(candidate['cfg']['img_hw']),
        'num_rover_classes': len(candidate['rover_vocab']),
        'val_probs_path': str(candidate_dir / 'val_probs.pt'),
        'test_probs_path': str(candidate_dir / 'test_probs.pt'),
        'val_shape': list(val_probs.shape),
        'test_shape': list(test_probs.shape),
    }

    save_json(meta, candidate_dir / 'meta.json')
    rtmdet_rows.append(meta)

    print('=' * 100)
    print(slug)
    print('val_probs:', tuple(val_probs.shape), '| test_probs:', tuple(test_probs.shape))

    del candidate['model']
    cleanup_cuda()

rtmdet_manifest_df = pd.DataFrame(rtmdet_rows)
rtmdet_manifest_df.to_csv(OUT_DIR / 'manifest_rtmdet_only.csv', index=False)
display(rtmdet_manifest_df)


{
  "checkpoint": "rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth",
  "raw_keys": 874,
  "backbone_candidate_keys": 458,
  "remapped_keys": 216,
  "loaded_keys": 458,
  "missing_keys": 0,
  "unexpected_keys": 0
}
loaded v7_rtmdet::best | model_type=v7_rtmdet_cspnext_lss_bev_cleaned | missing=0 unexpected=0


collect test probs -> test_probs.pt: 100%|██████████| 2000/2000 [04:12<00:00,  7.93it/s]


v7_rtmdet__best
val_probs: (220, 1, 188, 126) | test_probs: (2000, 1, 188, 126)
{
  "checkpoint": "rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth",
  "raw_keys": 874,
  "backbone_candidate_keys": 458,
  "remapped_keys": 216,
  "loaded_keys": 458,
  "missing_keys": 0,
  "unexpected_keys": 0
}
loaded v7_rtmdet::ema_best | model_type=v7_rtmdet_cspnext_lss_bev_cleaned | missing=0 unexpected=0


collect test probs -> test_probs.pt: 100%|██████████| 2000/2000 [04:12<00:00,  7.92it/s]


v7_rtmdet__ema_best
val_probs: (220, 1, 188, 126) | test_probs: (2000, 1, 188, 126)


,slug,run_name,ckpt_kind,model_type,ckpt_path,img_hw,num_rover_classes,val_probs_path,test_probs_path,val_shape,test_shape
0,v7_rtmdet__best,v7_rtmdet,best,v7_rtmdet_cspnext_lss_bev_cleaned,/home/jupyter/project/runs/v7_rtmdet_cspnext_l...,"[384, 704]",26,/home/jupyter/project/ensemble_infer_outputs_v...,/home/jupyter/project/ensemble_infer_outputs_v...,"[220, 1, 188, 126]","[2000, 1, 188, 126]"
1,v7_rtmdet__ema_best,v7_rtmdet,ema_best,v7_rtmdet_cspnext_lss_bev_cleaned,/home/jupyter/project/runs/v7_rtmdet_cspnext_l...,"[384, 704]",26,/home/jupyter/project/ensemble_infer_outputs_v...,/home/jupyter/project/ensemble_infer_outputs_v...,"[220, 1, 188, 126]","[2000, 1, 188, 126]"


### Что скачать на локалку после инференса

Скачай всю папку `ensemble_infer_outputs_v6_v7/`, в ней будут:
- `manifest.csv` и `manifest.json` — сводка по всем 4 кандидатам
- `val_info_export.csv` — порядок `val` объектов
- `test_info_export.csv` и `test_info_official.csv` — порядок `test` и официальный `info.csv`
- для каждого кандидата отдельная папка:
  - `val_probs.pt` — содержит `{'probs', 'gt'}`
  - `test_probs.pt`
  - `threshold_sweep.csv`
  - `meta.json`
- `single_model_submissions/` — sanity zip-посылки из лучших одиночных моделей

Для локального CPU-ансамбля тебе достаточно:
- все `val_probs.pt`
- все `test_probs.pt`
- `manifest.csv`
- `test_info_official.csv`
